# ============================================
# PROJETO 3 — AGENTE DE RH COM RAG + RERANKING
# LangChain + Streamlit
# ============================================

####Install

In [0]:
%restart_python

In [0]:
%pip uninstall -y langgraph langgraph-checkpoint langgraph-prebuilt langgraph-sdk

In [0]:
%pip install streamlit
%pip install langchain-community
%pip install langchain
%pip show typing_extensions
%pip show langchain-openai
%pip show openai
%pip show langchain-core
%pip install -U \
langchain \
langchain-core \
langchain-community \
langchain-openai \
openai
%pip list | grep -E "langchain|langgraph|pydantic|typing"

#### Biblioteca

In [0]:
import langchain
import langchain_community
import langchain_openai
import openai
import streamlit
import os
import streamlit as st

In [0]:
# ============================================
# PROJETO 3 — AGENTE DE RH COM RAG + RERANKING
# LangChain + Streamlit
# ============================================

# =========================
# 1. IMPORTAÇÕES
# =========================

# Injeta a chave como variável de ambiente
os.environ["OPENAI_API_KEY"] = "chave"

# Loaders e chunking
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings e LLM
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Vector Store
from langchain_community.vectorstores import Chroma

# Prompt
from langchain_core.prompts import PromptTemplate


# =========================
# 2. CONFIGURAÇÕES GERAIS
# =========================

# Diretório do banco vetorial
PERSIST_DIRECTORY = "./chroma_rh"

# Modelo de embeddings
EMBEDDING_MODEL = "text-embedding-3-small"

# Modelo de linguagem
LLM_MODEL = "gpt-4o-mini"



In [0]:
# =========================
# 3. LEITURA DOS DOCUMENTOS
# =========================

st.cache_data
def carregar_documentos():
    """
    Carrega os PDFs de políticas internas de RH
    """
    caminhos = [
        "/Workspace/Repos/patriciatamiresdesousa@gmail.com/arquitetura_langchain_tecnicas_avancadas_de_rag/politica_ferias.pdf",
        "/Workspace/Repos/patriciatamiresdesousa@gmail.com/arquitetura_langchain_tecnicas_avancadas_de_rag/politica_home_office.pdf",
        "/Workspace/Repos/patriciatamiresdesousa@gmail.com/arquitetura_langchain_tecnicas_avancadas_de_rag/politica_home_office.pdf"
    ]

    documentos = []

    for caminho in caminhos:
        loader = PyPDFLoader(caminho)
        docs = loader.load()

        for doc in docs:
            doc.metadata["documento"] = caminho

        documentos.extend(docs)

    return documentos


In [0]:

# =========================
# 4. CHUNKING
# =========================

def gerar_chunks(documentos):
    """
    Divide os documentos em chunks semânticos
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=150
    )

    return splitter.split_documents(documentos)

# =========================
# 5. ENRIQUECIMENTO COM METADADOS
# =========================

def enriquecer_chunks(chunks):
    """
    Classifica os chunks por categoria semântica
    """
    for chunk in chunks:
        texto = chunk.page_content.lower()

        if "férias" in texto:
            chunk.metadata["categoria"] = "ferias"
        elif "home office" in texto or "remoto" in texto:
            chunk.metadata["categoria"] = "home_office"
        elif "conduta" in texto or "ética" in texto:
            chunk.metadata["categoria"] = "conduta"
        else:
            chunk.metadata["categoria"] = "geral"

    return chunks